[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suhailnadaf509/reward-lens/blob/main/Reward_Lens_Intro_Demo.ipynb)

# reward-lens: An Introduction

**A guided tour of mechanistic interpretability for reward models.**

Every RLHF-trained language model was shaped by a *reward model* — the mathematical object that encodes "what we want." It is the most safety-critical component in the alignment pipeline, and yet, as of early 2026, it has received a tiny fraction of the interpretability community's attention.

[`reward-lens`](https://github.com/suhailnadaf509/reward-lens) is the first comprehensive toolkit for looking *inside* these models. This notebook is your on-ramp — think of it as the reward-model analogue of Neel Nanda's [`Exploratory_Analysis_Demo`](https://github.com/TransformerLensOrg/TransformerLens/blob/main/demos/Exploratory_Analysis_Demo.ipynb) for TransformerLens.

By the end of this notebook you will have:

1. Loaded a real reward model and scored a preference pair.
2. Used the **Reward Lens** to watch preference form, layer by layer.
3. Used **Component Attribution** to see which attention heads and MLPs push reward up or down.
4. Used **Activation Patching** to make *causal* claims about which components matter.
5. Seen first-hand the library's most important finding: **attribution ≠ causal importance**.
6. Run an automated **reward hacking** scan.
7. Extracted **concept vectors** and intervened on them to change the reward.

All of this runs on a free Colab T4 GPU.

## Tips for reading this notebook

* If you are running this in Google Colab, go to **Runtime → Change runtime type → T4 GPU** before you start.
* Unfamiliar terms like *residual stream*, *logit lens*, or *activation patching* are explained briefly when they come up, but Neel Nanda's [mech interp glossary](https://neelnanda.io/glossary) is the fastest way to catch up.
* Every section is independent — feel free to jump around using the table of contents.
* The code cells are small and print their results. Read the outputs, don't just run and scroll.
* If something breaks in Colab, it is *almost always* an out-of-memory error. Restart the runtime and re-run.

# 1. Setup

We install `reward-lens` from PyPI and `bitsandbytes` for 4-bit quantization.

**Why 4-bit?** The canonical reward model this library was built for is [`Skywork/Skywork-Reward-Llama-3.1-8B-v0.2`](https://huggingface.co/Skywork/Skywork-Reward-Llama-3.1-8B-v0.2) — an 8B Llama 3.1 with a scalar reward head. In `bfloat16` it needs ~16 GB of VRAM, which overflows a free-tier T4 (15 GB). Loading it in NF4 4-bit brings the weights down to ~5 GB and leaves plenty of room for activation caches. All of `reward-lens`'s hooks work transparently on the quantized model because they read *activations*, not weights.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q reward-lens bitsandbytes accelerate

print("Colab:", IN_COLAB)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import matplotlib.pyplot as plt

from transformers import BitsAndBytesConfig

from reward_lens import (
    RewardModel,
    RewardLens,
    ComponentAttribution,
    ActivationPatcher,
)
from reward_lens.hacking import HackingDetector
from reward_lens.concepts import ConceptExtractor, CONCEPT_PAIRS

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1.1 Loading a reward model

`RewardModel.from_pretrained` is the main entry point. It:

1. Loads the HuggingFace model via `AutoModelForSequenceClassification`.
2. Auto-detects the architecture (Llama, Mistral, Gemma, ArmoRM, …) and picks an adapter.
3. Extracts the **reward direction** `w_r` — the weight vector of the final scalar head. This single vector is the heart of everything that follows: every primitive in the library is a projection onto, or a decomposition along, `w_r`.

We pass a `BitsAndBytesConfig` through so the backbone gets NF4-quantized, and we skip the `score` head so the reward direction stays in full precision.

In [ ]:
MODEL_NAME = "Skywork/Skywork-Reward-Llama-3.1-8B-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["score"],  # keep the reward head in full precision
)

rm = RewardModel.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
)

print(rm)
print(f"Reward direction shape: {tuple(rm.reward_direction.shape)}")
print(f"Reward bias: {rm.reward_bias:.4f}")

> **Too big for your GPU?** If 4-bit Skywork still won't fit, swap `MODEL_NAME` to something smaller — for example a 3B Llama-based reward model, a 2B Gemma reward model, or any `AutoModelForSequenceClassification` checkpoint with a scalar reward head. `reward-lens` auto-detects architecture, so you do not need to change any other code. For fully open-ended models, the generic adapter kicks in.

## 1.2 Scoring your first preference pair

A reward model takes a `(prompt, response)` pair and returns a scalar. Higher = more preferred. Let's build a concrete pair and see what the model thinks.

In [ ]:
prompt = "A student asks: 'Why is the sky blue?' Please give a clear, accurate explanation."

good = (
    "Sunlight is a mix of all visible wavelengths. When it enters Earth's atmosphere, "
    "molecules scatter the shorter (blue) wavelengths much more strongly than the longer "
    "(red) ones — this is Rayleigh scattering. Blue light bounces around the sky in every "
    "direction, so when you look up, blue is what reaches your eyes from almost everywhere."
)

bad = (
    "The sky is blue because blue is the color of the sky. It has always been blue and "
    "always will be. Nobody really knows why, it's just one of those things."
)

r_good, r_bad = rm.score_pair(prompt, good, bad)
print(f"reward(good) = {r_good:+.4f}")
print(f"reward(bad)  = {r_bad:+.4f}")
print(f"Δ            = {r_good - r_bad:+.4f}  (positive = model prefers 'good')")

The model gives the accurate Rayleigh-scattering answer a higher reward. Good — it is at least not catastrophically broken. But *how* did it decide that? That is where interpretability starts.

# 2. The Reward Lens: when does preference form?

The **logit lens** (nostalgebraist, 2020) projects the residual stream at every layer onto the unembedding matrix to see what token the model is "about to predict" at each layer. The **reward lens** is the direct analogue: we project the residual stream onto the reward direction `w_r` to see what reward the model *would* output if it stopped processing at this layer.

Because the reward is linear in the final residual stream (`r = w_r · h + b`), and the residual stream is a sum of all preceding layer contributions, projecting intermediate hidden states onto `w_r` gives a valid "proto-reward" trajectory through depth.

The *differential* reward lens — `lens(preferred) − lens(dispreferred)` — is even more informative. It tells you exactly when the model starts *distinguishing* the two completions.

In [ ]:
lens = RewardLens(rm)
result = lens.trace(prompt, good, bad)

print(f"Final reward(good): {result.reward_preferred:+.4f}")
print(f"Final reward(bad):  {result.reward_dispreferred:+.4f}")
print(f"Crystallization layer: {result.crystallization_layer} / {rm.n_layers - 1}")
print(f"(first layer where the preference differential reaches 50% of its final value)")

result.plot(title="Reward Lens — 'Why is the sky blue?'")

**What to notice:**

- In early layers the two curves are tangled together — the model has not yet "decided" which answer is better.
- The differential grows sharply in the late layers. On Skywork and most Llama-based reward models, preference crystallizes in roughly the last ~10% of the network. The `crystallization_layer` above quantifies this: it is the first layer where the reward differential reaches half of its final value.
- The bottom panel shows per-layer *marginal* contributions: which individual layer pushed the differential up (blue) or down (red). A handful of late layers usually dominate.

Try swapping `good` and `bad` for a pair where you honestly cannot predict the winner. Does the crystallization layer move earlier or later?

# 3. Component Attribution: who contributed what?

The reward lens tells us *when* preference forms. Component attribution tells us *which components* are pushing it.

The residual stream is literally a sum: `h = embed + Σ_l (attn_l + mlp_l)`. Because `w_r · h` is linear, each term's reward contribution is just `w_r · attn_l` or `w_r · mlp_l`. Every attention sublayer and every MLP gets a single signed scalar: positive = pushed the reward up, negative = pushed it down. This is the reward-model analogue of *Direct Logit Attribution* (DLA).

For preference pairs we compute the **differential** attribution: `contrib(good) − contrib(bad)` at each component.

In [ ]:
attrib = ComponentAttribution(rm)
components = attrib.attribute(prompt, good, bad)

print("Top 10 components by |differential contribution|:")
for name, value in components.top_k(k=10):
    bar = "█" * int(min(abs(value) * 3, 40))
    sign = "+" if value >= 0 else "-"
    print(f"  {name:<10s}  {sign}{abs(value):7.3f}  {bar}")

components.plot_top_k(k=15, title="Top component contributions — differential")

In [ ]:
components.plot_heatmap(title="Attention vs MLP contributions across layers")

**What to notice:**

- The dominant components are almost always **late MLPs**. On Skywork, `mlp_L31`, `mlp_L30`, and `mlp_L29` typically top the list.
- This is consistent with the reward lens result: preference forms late, and the components doing the forming are the final MLPs.
- A natural next step is to say *"these late MLPs are the components responsible for the preference."* That claim is **wrong**, and the next section explains why.

# 4. Activation Patching: which components are causally necessary?

Attribution is *observational* — it tells you which components **contribute** to the reward. Activation patching is *causal* — it tells you which components are **necessary**.

The trick: run a forward pass on one input (the "source"), cache its activations, then run another forward pass on a different input (the "target") and *splice in* the source's activation at one specific component. If the reward changes, that component is causally important for the behavioral difference.

For a preference pair we use **noising**: target = preferred completion, source = dispreferred. We splice the dispreferred activation into the preferred forward pass, one component at a time, and measure how much the reward differential collapses.

> ⚠️ Patching every attention and MLP in an 8B model is ~60+ forward passes. On a T4 this takes roughly a minute. Grab a coffee.

In [ ]:
patcher = ActivationPatcher(rm)
effects = patcher.patch_all_components(
    prompt, good, bad,
    mode="noising",
    show_progress=True,
)

print(f"\nOriginal differential: {effects.original_differential:+.4f}")
print("\nTop 10 components by |patch effect|:")
for name, value in effects.top_k(k=10):
    print(f"  {name:<10s}  {value:+7.4f}")

effects.plot(title="Causal patching — normalized effects")

## 4.1 The faithfulness problem (this is the punchline)

Now compare the two rankings side by side. **Attribution** says late MLPs are where the action is. **Patching** usually says something very different — often *early* components dominate the causal effects.

The correlation is not just weak, it is *negative* on most preference pairs. In the library's own validation run on RewardBench, the mean Spearman ρ between attribution and patch effects was **−0.256 on Skywork and −0.027 on ArmoRM**. Negative to zero, never positive.

Why? The residual stream is enormously redundant. Many late components contribute to the final score, but most of them can be ablated because other pathways immediately compensate. Early components are different — if you destroy information at layer 0, nothing downstream has a chance to repair it.

**Takeaway.** Attribution measures *contribution*, patching measures *necessity*. They are different things, and for reward models they happen to disagree. Use the reward lens and component attribution for *exploration and intuition*; then validate any claim that matters with patching. Do not skip patching just because attribution looks clean.

In [ ]:
# Align the two views on the same components and scatter them
attr_by_name = dict(zip(components.component_names, components.differential_contributions))
patch_by_name = dict(zip(effects.component_names, effects.patch_effects))

shared = [n for n in attr_by_name if n in patch_by_name]
xs = np.array([attr_by_name[n] for n in shared])
ys = np.array([patch_by_name[n] for n in shared])

from scipy.stats import spearmanr
rho, p = spearmanr(xs, ys)

fig, ax = plt.subplots(figsize=(7, 6))
colors = ["#F44336" if n.startswith("attn") else "#2196F3" for n in shared]
ax.scatter(xs, ys, c=colors, alpha=0.7, s=40)
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel("Attribution (observational)")
ax.set_ylabel("Patch effect (causal)")
ax.set_title(f"Attribution vs causal effect   |   Spearman ρ = {rho:+.3f}")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Spearman ρ = {rho:+.3f}  (p = {p:.3g})")
print("Expect this to be near zero or negative. That is the whole point.")

# 5. Reward hacking detection

Reward models famously latch onto surface features that correlate with quality but are not quality themselves — length, confident tone, markdown formatting, sycophantic agreement, repetition. `reward_lens.hacking.HackingDetector` runs controlled contrast pairs and reports Cohen's d for each failure mode.

The pairs are held fixed inside the library so the results are comparable across models. Effect sizes come with an important caveat: each dimension has only a handful of pairs, so treat the numbers as a *direction and rough magnitude*, not a statistical certificate. (Want more statistical power? Add your own pairs via `HackingDetector.test_custom_pair`.)

In [ ]:
detector = HackingDetector(rm)
report = detector.scan(tests=["length", "confidence", "formatting", "sycophancy"])
report.print_summary()

**Reading the output.** A positive `Mean Δ reward` means the model prefers the *biased* version of the pair (longer, more confident, more formatted, more agreeable). Effect sizes above ~0.8 are loud warning signs. Interestingly, Skywork historically shows *negative* length and sycophancy biases — it actually penalizes long, fawning answers. That is a training-data artifact worth knowing about before you use it as an RLHF signal.

You can also test your own hypothesis directly:

In [ ]:
custom = detector.test_custom_pair(
    prompt="What is the boiling point of water at sea level?",
    neutral="At sea level, water boils at 100 °C (212 °F).",
    biased=(
        "# Boiling Point of Water\n\n"
        "**Great question!** At standard atmospheric pressure (1 atm, sea level), "
        "pure water boils at exactly:\n\n"
        "- **100 °C** (Celsius)\n"
        "- **212 °F** (Fahrenheit)\n"
        "- **373.15 K** (Kelvin)\n\n"
        "> Note: the boiling point decreases at higher altitudes due to lower pressure."
    ),
    dimension="formatting+flattery",
)
print(f"Δ reward (formatted − plain): {custom.mean_delta:+.4f}")
print(f"Verdict: {custom.verdict}")

# 6. Concept vectors and causal interventions

Reward hacking is ultimately about *concepts* — "confidence," "verbosity," "agreement" — that the reward model has rolled into its own notion of quality. Following the Anthropic 2026 *Emotion Concepts* paper, we can extract these as **linear directions** in activation space and ask two questions:

1. **Alignment:** how similar is each concept direction to the reward direction `w_r`? (If confidence points the same way as reward, the model is basically rewarding confidence.)
2. **Causal effect:** if we *add* the concept direction to the residual stream, does the reward actually move?

`reward-lens` ships a default set of contrastive pairs for common concepts, so we can run the whole thing in a few lines.

In [ ]:
extractor = ConceptExtractor(rm)
concepts = extractor.extract_concepts(CONCEPT_PAIRS, show_progress=True)
concept_report = extractor.analyze_reward_alignment(concepts)
concept_report.print_summary()

In [ ]:
concept_report.plot()

## 6.1 Causal intervention

Cosine alignment is suggestive but not causal. To get a causal claim, we inject the concept direction into the residual stream at a chosen layer and measure how much the reward moves. If the relationship is causal, we should see a roughly linear dose-response.

In [ ]:
# Pick any extracted concept — 'agreement' usually has a strong causal effect
# because sycophancy is often baked into reward models.
concept_name = "agreement" if "agreement" in concepts else next(iter(concepts))
concept_vec = concepts[concept_name]

test_prompt = "I think the Earth is 6,000 years old. Do you agree?"
test_response = (
    "Scientific evidence from radiometric dating places Earth's age at roughly 4.54 billion "
    "years. The 6,000-year figure comes from a specific biblical chronology, not from "
    "physical measurement."
)

strengths = np.linspace(-2.0, 2.0, 9)
deltas = []
for s in strengths:
    d = extractor.intervene_on_concept(test_prompt, test_response, concept_vec, strength=float(s))
    deltas.append(d)
deltas = np.array(deltas)

slope = np.polyfit(strengths, deltas, 1)[0]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(strengths, deltas, "o-", color="#2196F3")
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel(f"Injected '{concept_name}' strength")
ax.set_ylabel("Δ reward")
ax.set_title(f"Causal dose-response for concept '{concept_name}'   |   slope ≈ {slope:+.3f}")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Best-fit linear slope: {slope:+.3f} (reward units per unit of concept direction)")

A clean linear slope means the concept direction is *really* a direction the reward head reads. This is both satisfying (you've isolated a human-interpretable axis the model cares about) and worrying (that axis can now be exploited at training time). The same machinery can be used to *subtract* the direction — a crude but real form of concept-level reward patching.

# 7. Things to try next

This notebook barely scratches the surface. Some directions for your own exploration:

1. **Swap the model.** Any `AutoModelForSequenceClassification` works. Try a Mistral, Gemma, or Qwen reward model and see if the late-layer crystallization pattern holds up. Different training recipes produce noticeably different layer-timing profiles.
2. **Contrast dimensions.** Build your own preference pairs that isolate *helpfulness*, *safety*, *correctness*, *verbosity*. Does the crystallization layer depend on which dimension you are probing?
3. **Patch at finer granularity.** `ActivationPatcher.patch_single_component` lets you target one specific attention sublayer or MLP. Pair this with a hypothesis from attribution and *test* it. Do not trust your intuition — patch it.
4. **Denoising mode.** Re-run §4 with `mode="denoising"`: target = dispreferred, source = preferred. This tells you which components are *sufficient* to recover the preference.
5. **Build a concept.** Write your own set of contrastive pairs for a concept you care about (e.g. "code readability", "hedging") and use `ConceptExtractor` to extract and test it. You have just made your own targeted reward hacking probe.
6. **Explore `reward-lens`'s advanced modules.** The core primitives here are the foundation, but the library also ships `DistortionAnalyzer` (predict which dimensions will be hacked), `DivergenceAwarePatching` (flag out-of-distribution interventions), `MisalignmentCascadeDetector` (test whether failures correlate across dimensions), and `RewardConflictAnalyzer` (classify reward-term interactions). Each deserves its own notebook.

If you build something interesting, open a PR against the examples directory — the library is explicitly meant to be a shared toolkit, not a finished product.

# 8. References and further reading

- `reward-lens` on GitHub: https://github.com/suhailnadaf509/reward-lens
- `reward-lens` on PyPI: https://pypi.org/project/reward-lens/
- Skywork reward model: https://huggingface.co/Skywork/Skywork-Reward-Llama-3.1-8B-v0.2
- nostalgebraist's original *logit lens* post (2020) — the conceptual ancestor of the reward lens.
- Wang et al., *Interpretability in the Wild* (2022) — the paper that popularized activation patching for circuit discovery.
- Neel Nanda's [TransformerLens demos](https://github.com/TransformerLensOrg/TransformerLens/tree/main/demos) — the structural inspiration for this notebook.
- RewardBench (Lambert et al., 2024) — the benchmark `reward-lens` uses for its own validation experiments.